In [1]:
!pip install groq pdfplumber chromadb sentence-transformers gradio numpy
!pip install google-generativeai
!pip install python-dotenv

In [2]:
with open('.env', 'w') as f:
    f.write('GROQ_API_KEY=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\n')
    f.write('EMBEDDING_MODEL=all-MiniLM-L6-v2\n')
    f.write('CHUNK_SIZE=500\n')
    f.write('CHUNK_OVERLAP=100\n')
    f.write('TOP_K_RESULTS=5\n')
    f.write('LLM_MODEL=llama-3.1-8b-instant\n')

print(".env file created")

.env file created


In [3]:
import os
import uuid
import numpy as np
import pdfplumber
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import gradio as gr
from typing import List, Dict, Optional
from dotenv import load_dotenv

load_dotenv()

print("All imports successful")

All imports successful


In [4]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please check your .env file.")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", 500))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", 100))
TOP_K_RESULTS = int(os.getenv("TOP_K_RESULTS", 5))
COLLECTION_NAME = "pdf_chatbot"
LLM_MODEL = os.getenv("LLM_MODEL", "llama-3.1-8b-instant")

print("Configuration loaded from environment")
print(f"LLM: {LLM_MODEL} via Groq (free)")
print(f"Embeddings: {EMBEDDING_MODEL} (runs locally, free)")
print(f"Vector DB: ChromaDB (local, free)")

Configuration loaded from environment
LLM: llama-3.1-8b-instant via Groq (free)
Embeddings: all-MiniLM-L6-v2 (runs locally, free)
Vector DB: ChromaDB (local, free)


In [4]:
class PDFProcessor:
    """
    Extracts text from PDF files while tracking page numbers.
    Page numbers are critical for source citations in answers.
    """

    def extract_text_with_pages(self, pdf_path: str) -> List[Dict]:
        pages = []
        filename = os.path.basename(pdf_path)

        try:
            with pdfplumber.open(pdf_path) as pdf:
                total_pages = len(pdf.pages)
                print(f"'{filename}' has {total_pages} pages")

                for page_num, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text()

                    if text and text.strip():
                        pages.append({
                            "page_number": page_num,
                            "text": text.strip(),
                            "source": filename
                        })
                    else:
                        print(f"Page {page_num} appears blank or unreadable")

        except Exception as e:
            print(f"Error reading '{filename}': {e}")

        print(f"Extracted {len(pages)} pages with content")
        return pages

processor = PDFProcessor()
print("PDFProcessor initialized")

PDFProcessor initialized


In [6]:
# This creates the complete app.py file with UTF-8 encoding
app_content = '''import os
import uuid
import numpy as np
import pdfplumber
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import gradio as gr
from typing import List, Dict, Optional
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configuration
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment variables")

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", 500))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", 100))
TOP_K_RESULTS = int(os.getenv("TOP_K_RESULTS", 5))
COLLECTION_NAME = "pdf_chatbot"
LLM_MODEL = os.getenv("LLM_MODEL", "llama-3.1-8b-instant")

class PDFProcessor:
    def extract_text_with_pages(self, pdf_path: str) -> List[Dict]:
        pages = []
        filename = os.path.basename(pdf_path)
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text()
                    if text and text.strip():
                        pages.append({
                            "page_number": page_num,
                            "text": text.strip(),
                            "source": filename
                        })
        except Exception as e:
            print(f"Error reading '{filename}': {e}")
        return pages

class TextChunker:
    def __init__(self, chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def chunk_pages(self, pages: List[Dict]) -> List[Dict]:
        all_chunks = []
        for page in pages:
            page_text = page["text"]
            sub_chunks = self._split_text(page_text)
            for idx, chunk_text in enumerate(sub_chunks):
                all_chunks.append({
                    "chunk_id": str(uuid.uuid4()),
                    "text": chunk_text,
                    "page_number": page["page_number"],
                    "source": page["source"],
                    "chunk_index": idx
                })
        return all_chunks

    def _split_text(self, text: str) -> List[str]:
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start += self.chunk_size - self.chunk_overlap
        return chunks

class VectorStore:
    def __init__(self):
        self.embedder = SentenceTransformer(EMBEDDING_MODEL)
        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )

    def add_chunks(self, chunks: List[Dict]):
        if not chunks:
            return
        texts = [chunk["text"] for chunk in chunks]
        embeddings = self.embedder.encode(texts, show_progress_bar=False, batch_size=32).tolist()
        self.collection.add(
            ids=[chunk["chunk_id"] for chunk in chunks],
            embeddings=embeddings,
            documents=texts,
            metadatas=[{
                "page_number": chunk["page_number"],
                "source": chunk["source"],
                "chunk_index": chunk["chunk_index"]
            } for chunk in chunks]
        )

    def search(self, query: str, top_k: int = TOP_K_RESULTS) -> List[Dict]:
        query_embedding = self.embedder.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=min(top_k, self.collection.count())
        )
        retrieved_chunks = []
        for i in range(len(results["ids"][0])):
            retrieved_chunks.append({
                "text": results["documents"][0][i],
                "page_number": results["metadatas"][0][i]["page_number"],
                "source": results["metadatas"][0][i]["source"],
                "relevance_score": 1 - results["distances"][0][i]
            })
        return retrieved_chunks

    def clear(self):
        self.chroma_client.delete_collection(COLLECTION_NAME)
        self.collection = self.chroma_client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )

    def count(self) -> int:
        return self.collection.count()

class PDFChatBot:
    SYSTEM_PROMPT = """You are an expert document analyst assistant.
Your job is to answer questions based ONLY on the provided PDF document excerpts.

Rules you must follow:
1. ONLY use information from the provided context, never use outside knowledge
2. ALWAYS cite sources: mention the document name and page number for every claim
3. If the answer is not in the provided context, say: I could not find this information in the uploaded documents
4. Be concise but complete, do not pad your answers
5. If multiple sources support a point, mention all of them
6. Use bullet points for clarity when listing multiple facts

Citation format: (Source: [filename], Page [X])"""

    def __init__(self, api_key: str):
        self.client = Groq(api_key=api_key)
        self.chat_history = []

    def _build_context_block(self, chunks: List[Dict]) -> str:
        context_parts = []
        for i, chunk in enumerate(chunks, 1):
            context_parts.append(
                f"[EXCERPT {i}]\\n"
                f"Document: {chunk['source']}\\n"
                f"Page: {chunk['page_number']}\\n"
                f"Relevance: {chunk['relevance_score']:.2%}\\n"
                f"Content:\\n{chunk['text']}"
            )
        return "\\n\\n" + "-" * 50 + "\\n\\n".join(context_parts)

    def ask(self, question: str, retrieved_chunks: List[Dict]) -> str:
        context = self._build_context_block(retrieved_chunks)
        user_message = f"""Here are the relevant excerpts from the uploaded PDF documents:

{context}

-------------------------------------------------

Based on the above excerpts, please answer this question:
{question}

Remember to cite the document name and page number for each claim."""

        self.chat_history.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model=LLM_MODEL,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT}
            ] + self.chat_history
        )
        answer = response.choices[0].message.content
        self.chat_history.append({"role": "assistant", "content": answer})
        return answer

    def clear_history(self):
        self.chat_history = []

# Initialize components
processor = PDFProcessor()
chunker = TextChunker()
vector_store = VectorStore()
chatbot = PDFChatBot(api_key=GROQ_API_KEY)
uploaded_file_names = []

def process_uploaded_pdfs(files) -> str:
    global uploaded_file_names
    if not files:
        return "No files selected. Please upload at least one PDF."
    
    vector_store.clear()
    chatbot.clear_history()
    uploaded_file_names = []
    all_chunks = []
    
    for file in files:
        pages = processor.extract_text_with_pages(file.name)
        if not pages:
            continue
        chunks = chunker.chunk_pages(pages)
        all_chunks.extend(chunks)
        uploaded_file_names.append(os.path.basename(file.name))
    
    if not all_chunks:
        return "Could not extract text from any uploaded PDFs."
    
    vector_store.add_chunks(all_chunks)
    status = (
        f"Ready. Processed {len(uploaded_file_names)} document(s):\\n"
        + "\\n".join(f"  - {name}" for name in uploaded_file_names)
        + f"\\n\\nTotal chunks in database: {vector_store.count()}"
        + f"\\nYou can now ask questions in the chat."
    )
    return status

def chat_response(user_message: str, history: list) -> tuple:
    if not user_message.strip():
        return history, "", ""
    
    if not uploaded_file_names:
        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": "Please upload and process a PDF first."})
        return history, "", ""
    
    retrieved = vector_store.search(user_message, top_k=TOP_K_RESULTS)
    
    if not retrieved:
        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": "No relevant content found. Try rephrasing your question."})
        return history, "", ""
    
    answer = chatbot.ask(user_message, retrieved)
    sources_text = format_sources(retrieved)
    
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": answer})
    
    return history, "", sources_text

def format_sources(chunks: List[Dict]) -> str:
    if not chunks:
        return "No sources retrieved."
    
    lines = ["### Sources Used\\n"]
    seen_pages = set()
    
    for i, chunk in enumerate(chunks, 1):
        key = f"{chunk['source']}_p{chunk['page_number']}"
        if key in seen_pages:
            continue
        seen_pages.add(key)
        
        lines.append(
            f"**[{i}] {chunk['source']} - Page {chunk['page_number']}**  \\n"
            f"Relevance: {chunk['relevance_score']:.0%}  \\n"
            f"Excerpt:  \\n"
            f"> {chunk['text'][:200].replace(chr(10), ' ')}...  \\n"
        )
    
    return "\\n".join(lines)

def clear_chat_history():
    chatbot.clear_history()
    return [], ""

# Create Gradio Interface
with gr.Blocks(title="PDF AI Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # PDF AI Chatbot
    Upload your PDF documents and ask questions. Get answers with page citations.
    """)
    
    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            gr.Markdown("## Upload Documents")
            file_input = gr.File(
                label="Select PDF files (up to 50MB each)",
                file_count="multiple",
                file_types=[".pdf"],
                height=150
            )
            process_btn = gr.Button("Process PDFs", variant="primary", size="lg")
            status_box = gr.Textbox(
                label="Processing Status",
                value="Upload PDFs and click Process PDFs to begin.",
                interactive=False,
                lines=6
            )
            gr.Markdown("---")
            gr.Markdown("""
            **Tips:**
            - Upload multiple PDFs at once
            - Ask follow-up questions naturally
            - Check Sources panel for citations
            - Use Clear Chat to start over
            """)
        
        with gr.Column(scale=2):
            gr.Markdown("## Chat with your Documents")
            chatbot_ui = gr.Chatbot(label="Conversation", height=400)
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Ask a question about your PDF...",
                    label="",
                    scale=5,
                    lines=1,
                    max_lines=3
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)
            with gr.Row():
                clear_btn = gr.Button("Clear Chat", variant="secondary", scale=1)
    
    gr.Markdown("---")
    gr.Markdown("## Retrieved Sources and Excerpts")
    sources_display = gr.Markdown(value="Sources will appear here after you ask a question.")
    
    # Event handlers
    process_btn.click(
        fn=process_uploaded_pdfs,
        inputs=[file_input],
        outputs=[status_box]
    )
    
    send_btn.click(
        fn=chat_response,
        inputs=[msg_box, chatbot_ui],
        outputs=[chatbot_ui, msg_box, sources_display]
    )
    
    msg_box.submit(
        fn=chat_response,
        inputs=[msg_box, chatbot_ui],
        outputs=[chatbot_ui, msg_box, sources_display]
    )
    
    clear_btn.click(
        fn=clear_chat_history,
        outputs=[chatbot_ui, sources_display]
    )

# Launch the app
if __name__ == "__main__":
    demo.launch(share=True, debug=False, show_error=True)
'''

# Write with UTF-8 encoding
with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_content)
print(" app.py created successfully with UTF-8 encoding!")

 app.py created successfully with UTF-8 encoding!


In [6]:
class VectorStore:
    """
    Manages embeddings and similarity search.
    Both the embedding model and ChromaDB are completely free and run locally.
    """

    def __init__(self):
        print("Loading embedding model (first time may take around 1 minute)...")
        self.embedder = SentenceTransformer(EMBEDDING_MODEL)
        print(f"Loaded: {EMBEDDING_MODEL}")

        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )
        print(f"ChromaDB collection ready")

    def add_chunks(self, chunks: List[Dict]):
        if not chunks:
            print("No chunks to add")
            return

        print(f"Embedding {len(chunks)} chunks...")

        texts = [chunk["text"] for chunk in chunks]
        embeddings = self.embedder.encode(
            texts,
            show_progress_bar=True,
            batch_size=32
        ).tolist()

        self.collection.add(
            ids=[chunk["chunk_id"] for chunk in chunks],
            embeddings=embeddings,
            documents=texts,
            metadatas=[{
                "page_number": chunk["page_number"],
                "source": chunk["source"],
                "chunk_index": chunk["chunk_index"]
            } for chunk in chunks]
        )

        print(f"Stored {len(chunks)} chunks in vector database")

    def search(self, query: str, top_k: int = TOP_K_RESULTS) -> List[Dict]:
        query_embedding = self.embedder.encode([query]).tolist()

        results = self.collection.query(
            query_embeddings=query_embedding,
            n_results=min(top_k, self.collection.count())
        )

        retrieved_chunks = []
        for i in range(len(results["ids"][0])):
            retrieved_chunks.append({
                "text": results["documents"][0][i],
                "page_number": results["metadatas"][0][i]["page_number"],
                "source": results["metadatas"][0][i]["source"],
                "relevance_score": 1 - results["distances"][0][i]
            })

        return retrieved_chunks

    def clear(self):
        self.chroma_client.delete_collection(COLLECTION_NAME)
        self.collection = self.chroma_client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )
        print("Vector store cleared")

    def count(self) -> int:
        return self.collection.count()

vector_store = VectorStore()

Loading embedding model (first time may take around 1 minute)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded: all-MiniLM-L6-v2
ChromaDB collection ready


In [7]:
class PDFChatBot:
    """
    Uses Groq (free) with Llama 3.1 to answer questions.
    Groq API is OpenAI-compatible, making it easy to use.
    """

    SYSTEM_PROMPT = """You are an expert document analyst assistant.
Your job is to answer questions based ONLY on the provided PDF document excerpts.

Rules you must follow:
1. ONLY use information from the provided context, never use outside knowledge
2. ALWAYS cite sources: mention the document name and page number for every claim
3. If the answer is not in the provided context, say: I could not find this information in the uploaded documents
4. Be concise but complete, do not pad your answers
5. If multiple sources support a point, mention all of them
6. Use bullet points for clarity when listing multiple facts

Citation format: (Source: [filename], Page [X])"""

    def __init__(self, api_key: str):
        # Groq client - completely free
        self.client = Groq(api_key=api_key)
        self.chat_history = []

    def _build_context_block(self, chunks: List[Dict]) -> str:
        context_parts = []
        for i, chunk in enumerate(chunks, 1):
            context_parts.append(
                f"[EXCERPT {i}]\n"
                f"Document: {chunk['source']}\n"
                f"Page: {chunk['page_number']}\n"
                f"Relevance: {chunk['relevance_score']:.2%}\n"
                f"Content:\n{chunk['text']}"
            )
        return "\n\n" + "-" * 50 + "\n\n".join(context_parts)

    def ask(self, question: str, retrieved_chunks: List[Dict]) -> str:
        context = self._build_context_block(retrieved_chunks)

        user_message = f"""Here are the relevant excerpts from the uploaded PDF documents:

{context}

-------------------------------------------------

Based on the above excerpts, please answer this question:
{question}

Remember to cite the document name and page number for each claim."""

        # Add to history
        self.chat_history.append({
            "role": "user",
            "content": user_message
        })

        # Groq API call (free)
        # This uses the same OpenAI-compatible format
        response = self.client.chat.completions.create(
            model=LLM_MODEL,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT}
            ] + self.chat_history
        )

        answer = response.choices[0].message.content

        self.chat_history.append({
            "role": "assistant",
            "content": answer
        })

        return answer

    def ask_streaming(self, question: str, retrieved_chunks: List[Dict]):
        """
        Streaming version - yields text token by token (bonus feature).
        Groq supports streaming natively and it is free.
        """
        context = self._build_context_block(retrieved_chunks)

        user_message = f"""Here are the relevant excerpts from the uploaded PDF documents:

{context}

-------------------------------------------------

Based on the above excerpts, please answer this question:
{question}

Remember to cite the document name and page number for each claim."""

        self.chat_history.append({"role": "user", "content": user_message})
        full_response = ""

        # Groq streaming API call (free)
        stream = self.client.chat.completions.create(
            model=LLM_MODEL,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT}
            ] + self.chat_history,
            stream=True
        )

        for chunk in stream:
            text = chunk.choices[0].delta.content
            if text is not None:
                full_response += text
                yield text

        self.chat_history.append({"role": "assistant", "content": full_response})

    def clear_history(self):
        self.chat_history = []
        print("Chat history cleared")

chatbot = PDFChatBot(api_key=GROQ_API_KEY)
print("Chatbot initialized with free Groq API")

Chatbot initialized with free Groq API


In [8]:
uploaded_file_names = []

def process_uploaded_pdfs(files) -> str:
    global uploaded_file_names

    if not files:
        return "No files selected. Please upload at least one PDF."

    vector_store.clear()
    chatbot.clear_history()
    uploaded_file_names = []
    all_chunks = []

    print(f"Processing {len(files)} file(s)...")

    for file in files:
        filename = os.path.basename(file.name)
        print(f"Processing: {filename}")

        pages = processor.extract_text_with_pages(file.name)

        if not pages:
            print(f"Could not extract text from {filename}")
            continue

        chunks = chunker.chunk_pages(pages)
        all_chunks.extend(chunks)
        uploaded_file_names.append(filename)

        print(f"{filename}: {len(pages)} pages -> {len(chunks)} chunks")

    if not all_chunks:
        return "Could not extract text from any uploaded PDFs."

    vector_store.add_chunks(all_chunks)

    status = (
        f"Ready. Processed {len(uploaded_file_names)} document(s):\n"
        + "\n".join(f"  - {name}" for name in uploaded_file_names)
        + f"\n\nTotal chunks in database: {vector_store.count()}"
        + f"\nYou can now ask questions in the chat."
    )
    return status


def chat_response(user_message: str, history: list) -> tuple:
    if not user_message.strip():
        return history, "", ""

    if not uploaded_file_names:
        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": "Please upload and process a PDF first."})
        return history, "", ""

    retrieved = vector_store.search(user_message, top_k=TOP_K_RESULTS)

    if not retrieved:
        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": "No relevant content found. Try rephrasing your question."})
        return history, "", ""

    answer = chatbot.ask(user_message, retrieved)
    sources_text = format_sources(retrieved)

    # Gradio 6.0 requires this dict format instead of [user, assistant] list
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": answer})

    return history, "", sources_text


def format_sources(chunks: List[Dict]) -> str:
    if not chunks:
        return "No sources retrieved."

    lines = ["### Sources Used\n"]
    seen_pages = set()

    for i, chunk in enumerate(chunks, 1):
        key = f"{chunk['source']}_p{chunk['page_number']}"
        if key in seen_pages:
            continue
        seen_pages.add(key)

        lines.append(
            f"**[{i}] {chunk['source']} - Page {chunk['page_number']}**  \n"
            f"Relevance: {chunk['relevance_score']:.0%}  \n"
            f"Excerpt:  \n"
            f"> {chunk['text'][:200].replace(chr(10), ' ')}...  \n"
        )

    return "\n".join(lines)


def clear_chat_history():
    chatbot.clear_history()
    return [], ""

print("Helper functions defined")

Helper functions defined


In [9]:
with gr.Blocks(title="PDF AI Chatbot") as demo:

    gr.Markdown("""
    # PDF AI Chatbot
    Upload your PDF documents and ask questions. Get answers with page citations.
    """)

    with gr.Row():

        with gr.Column(scale=1, min_width=300):
            gr.Markdown("## Upload Documents")

            file_input = gr.File(
                label="Select PDF files (up to 50MB each)",
                file_count="multiple",
                file_types=[".pdf"],
                height=150
            )

            process_btn = gr.Button(
                "Process PDFs",
                variant="primary",
                size="lg"
            )

            status_box = gr.Textbox(
                label="Processing Status",
                value="Upload PDFs and click Process PDFs to begin.",
                interactive=False,
                lines=6
            )

            gr.Markdown("---")
            gr.Markdown("""
            Tips:
            - Upload multiple PDFs at once
            - Ask follow-up questions naturally
            - Check Sources panel for citations
            - Use Clear Chat to start over
            """)

        with gr.Column(scale=2):
            gr.Markdown("## Chat with your Documents")

            chatbot_ui = gr.Chatbot(
                label="Conversation",
                height=400,
                show_label=True
            )

            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Ask a question about your PDF...",
                    label="",
                    scale=5,
                    lines=1,
                    max_lines=3
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

            with gr.Row():
                clear_btn = gr.Button("Clear Chat", variant="secondary", scale=1)
                gr.Markdown("Press Enter or click Send", scale=3)

    gr.Markdown("---")
    gr.Markdown("## Retrieved Sources and Excerpts")
    sources_display = gr.Markdown(
        value="Sources will appear here after you ask a question."
    )

    process_btn.click(
        fn=process_uploaded_pdfs,
        inputs=[file_input],
        outputs=[status_box]
    )

    send_btn.click(
        fn=chat_response,
        inputs=[msg_box, chatbot_ui],
        outputs=[chatbot_ui, msg_box, sources_display]
    )

    msg_box.submit(
        fn=chat_response,
        inputs=[msg_box, chatbot_ui],
        outputs=[chatbot_ui, msg_box, sources_display]
    )

    clear_btn.click(
        fn=clear_chat_history,
        outputs=[chatbot_ui, sources_display]
    )

print("Gradio UI built successfully")

Gradio UI built successfully


In [10]:
demo.launch(
    share=True,
    debug=False,
    show_error=True
)


#demo.launch(
#    share=True,
#    server_name="0.0.0.0",
#    debug=True
#)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [11]:
print("=" * 60)
print("COMPONENT TEST SUITE")
print("=" * 60)

# Test 1: Chunking
print("\nTest 1: Text Chunking")
sample_pages = [
    {"page_number": 1, "text": "A" * 1200, "source": "test.pdf"},
    {"page_number": 2, "text": "B" * 300,  "source": "test.pdf"}
]
test_chunks = chunker.chunk_pages(sample_pages)
print(f"1200 + 300 chars -> {len(test_chunks)} chunks")

# Test 2: Embeddings
print("\nTest 2: Embeddings")
test_embed = vector_store.embedder.encode(["Hello world"])
print(f"Embedding shape: {test_embed.shape} (expected (1, 384))")

# Test 3: Vector Search
print("\nTest 3: Vector Search")
sample_chunks = [
    {"chunk_id": str(uuid.uuid4()), "text": "The sky is blue.", "page_number": 1, "source": "test.pdf", "chunk_index": 0},
    {"chunk_id": str(uuid.uuid4()), "text": "Python is a programming language.", "page_number": 2, "source": "test.pdf", "chunk_index": 0},
]
vector_store.clear()
vector_store.add_chunks(sample_chunks)
results = vector_store.search("What color is the sky?", top_k=1)
print(f"Query result: '{results[0]['text']}' (Page {results[0]['page_number']})")
assert "sky" in results[0]["text"].lower(), "Retrieval test failed"
print("Retrieval test passed")
vector_store.clear()

# Test 4: Groq API
print("\nTest 4: Groq API Connection (Free)")
test_client = Groq(api_key=GROQ_API_KEY)
test_response = test_client.chat.completions.create(
    model=LLM_MODEL,
    max_tokens=20,
    messages=[{"role": "user", "content": "Say: API connection works"}]
)
print(f"Groq says: {test_response.choices[0].message.content}")

print("\n" + "=" * 60)
print("ALL TESTS PASSED")
print("=" * 60)

COMPONENT TEST SUITE

Test 1: Text Chunking
1200 + 300 chars -> 4 chunks

Test 2: Embeddings
Embedding shape: (1, 384) (expected (1, 384))

Test 3: Vector Search
Vector store cleared
Embedding 2 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Stored 2 chunks in vector database
Query result: 'The sky is blue.' (Page 1)
Retrieval test passed
Vector store cleared

Test 4: Groq API Connection (Free)
Groq says: API connection has been established successfully.

ALL TESTS PASSED


In [12]:
# Generate requirements.txt for HuggingFace deployment

requirements = """groq
pdfplumber
chromadb
sentence-transformers
gradio
numpy
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt saved")
print("\nFor HuggingFace Spaces deployment:")
print("1. Go to https://huggingface.co and create a free account")
print("2. Create a new Space with Gradio SDK")
print("3. Upload app.py and requirements.txt")
print("4. Go to Settings -> Variables and Secrets")
print("5. Add secret: Name = GROQ_API_KEY, Value = your key")
print("6. Your app will be live at huggingface.co/spaces/YOUR_NAME/pdf-chatbot")

requirements.txt saved

For HuggingFace Spaces deployment:
1. Go to https://huggingface.co and create a free account
2. Create a new Space with Gradio SDK
3. Upload app.py and requirements.txt
4. Go to Settings -> Variables and Secrets
5. Add secret: Name = GROQ_API_KEY, Value = your key
6. Your app will be live at huggingface.co/spaces/YOUR_NAME/pdf-chatbot


C:\Users\Abhisek\Documents\python\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\Abhisek\Documents\python\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Vector store cleared
Chat history cleared
Processing 1 file(s)...
Processing: STATISTICS_PROBABILITY.pptx.pdf
'STATISTICS_PROBABILITY.pptx.pdf' has 62 pages
Extracted 62 pages with content
STATISTICS_PROBABILITY.pptx.pdf: 62 pages -> 63 chunks
Embedding 63 chunks...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Stored 63 chunks in vector database


C:\Users\Abhisek\Documents\python\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
